In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
import scipy as sp
import seaborn as sns

In [ ]:
in_dir = '../../results/01_process_access/15_plot_footprint'
out_dir = '../../results/01_process_access/16_estimate_footprint'
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# get motif width from PWM
df = pd.read_csv('~/rgtdata/motifs/jaspar_vertebrates.mtf', sep='\t', header=None, index_col=0)

motif_names, motif_width = [], []

for motif_name in df[1].tolist():
    pwm_filename = f'~/rgtdata/motifs/jaspar_vertebrates/{motif_name}.pwm'
    pwm = pd.read_csv(pwm_filename, sep=" ", header=None)
    
    assert pwm.shape[0] == 4, f"Expected 4 rows in PWM for {motif_name}, got {pwm.shape[0]}"
    
    motif_names.append(motif_name)
    motif_width.append(pwm.shape[1])
    
df_motif = pd.DataFrame({'motif_name': motif_names, 'motif_width': motif_width})

In [ ]:
def get_fp_score(signal: np.ndarray, motif_width: int = 5) -> pd.DataFrame:
    fp_width_list, fp_height_list = [], []

    for fp_width in range(motif_width, 100, 1):
        half_fp_width = fp_width // 2
        fp_signal = signal[500-half_fp_width:500+half_fp_width].mean()
        left_signal = signal[500-half_fp_width - fp_width:500-half_fp_width].mean()
        right_signal = signal[500+half_fp_width:500+half_fp_width + fp_width].mean()
        fp_height = (left_signal + right_signal) / 2 - fp_signal

        fp_width_list.append(fp_width)
        fp_height_list.append(fp_height)

    return pd.DataFrame({'fp_width': fp_width_list, 'fp_height': fp_height_list})

In [ ]:
# for testing
# df_motif = df_motif.head(30)

In [ ]:
motif_name_list = []
motif_width_list = []
cell_type_list = []
fp_height_list = []
fp_width_list = []
data_list = []

for cell_type in ['K562', 'HepG2']:
    os.makedirs(f'{out_dir}/{cell_type}', exist_ok=True)
    
    for motif_name, motif_width in zip(df_motif['motif_name'], df_motif['motif_width']):
        motif_name = motif_name.replace(" ", "_").replace("/", "_").replace("(", "").replace(")", "")
        
        df_access = pd.read_csv(f'{in_dir}/{cell_type}/ACCESS/{motif_name}.csv')
        df_atac = pd.read_csv(f'{in_dir}/{cell_type}/ATAC/{motif_name}.csv')

        # estimate the footprint characteristics
        df_fp_access = get_fp_score(df_access['signal'].values, motif_width=motif_width)
        df_fp_atac = get_fp_score(df_atac['signal'].values, motif_width=motif_width)
        
        max_height_access = df_fp_access['fp_height'].max()
        max_width_access = df_fp_access.loc[df_fp_access['fp_height'].idxmax(), 'fp_width']

        max_height_atac = df_fp_atac['fp_height'].max()
        max_width_atac = df_fp_atac.loc[df_fp_atac['fp_height'].idxmax(), 'fp_width']

        # store the results
        motif_name_list.append(motif_name)
        motif_name_list.append(motif_name)
        motif_width_list.append(motif_width)
        motif_width_list.append(motif_width)
        cell_type_list.append(cell_type)
        cell_type_list.append(cell_type)

        fp_height_list.append(max_height_access)
        fp_width_list.append(max_width_access)
        fp_height_list.append(max_height_atac)
        fp_width_list.append(max_width_atac)
        data_list.append("ACCESS")
        data_list.append("ATAC")

        # # plot the signal
        fig, axes = plt.subplots(2, 2, figsize=(10, 10))
        sns.lineplot(data=df_access[450:550], x='position', y='signal', ax=axes[0, 0])
        sns.lineplot(data=df_atac[450:550], x='position', y='signal', ax=axes[0, 1])
        sns.lineplot(data=df_fp_access, x='fp_width', y='fp_height', ax=axes[1, 0])
        sns.lineplot(data=df_fp_atac, x='fp_width', y='fp_height', ax=axes[1, 1])

        axes[1, 0].axvline(x=max_width_access, color='r', linestyle='--')
        axes[1, 1].axvline(x=max_width_atac, color='r', linestyle='--')

        axes[0, 0].set_title(f'ACCESS')
        axes[0, 1].set_title(f'ATAC')

        axes[1, 0].set_title(f'ACCESS: Max Height {max_height_access:.2f} at Width {max_width_access}')
        axes[1, 1].set_title(f'ATAC: Max Height {max_height_atac:.2f} at Width {max_width_atac}')
        
        plt.savefig(f"{out_dir}/{cell_type}/{motif_name}.png", dpi=300, bbox_inches='tight')
        plt.close(fig)

In [ ]:
df = pd.DataFrame({
    'motif_name': motif_name_list,
    'motif_width': motif_width_list,
    'fp_width': fp_width_list,
    'fp_height': fp_height_list,
    'data': data_list,
    'cell_type': cell_type_list})

In [ ]:
df.to_csv(f'{out_dir}/footprint.csv', index=False)

In [ ]:
sns.boxplot(x='cell_type', y='fp_height', hue='data', data=df, showfliers=False)
plt.ylabel('Footprint Height')

In [ ]:
# no outliers
sns.boxplot(x='cell_type', y='fp_width', hue='data', data=df, showfliers=False)
plt.ylabel('Footprint Width (nt)')

In [ ]:
# long to wide format
df_wide = df.pivot_table(index=['motif_name', 'cell_type'], 
                         columns='data', values=['fp_height']).reset_index()

# rename columns
df_wide.columns = ['motif_name', 'cell_type', 'ACCESS', 'ATAC']

df_wide.to_csv(f"{out_dir}/fp_height.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sns.scatterplot(data=df_wide, x='ATAC', y='ACCESS', ax=ax, hue='cell_type')
ax.set_xlabel('Footprint Height (ATAC)')
ax.set_ylabel('Footprint Height (ACCESS)')
plt.plot([df_wide[['ACCESS', 'ATAC']].min().min(), df_wide[['ACCESS', 'ATAC']].max().max()], 
         [df_wide[['ACCESS', 'ATAC']].min().min(), df_wide[['ACCESS', 'ATAC']].max().max()], 'r--')

In [ ]:
# long to wide format
df_wide = df.pivot_table(index=['motif_name', 'cell_type'], 
                         columns='data', values=['fp_width']).reset_index()

# rename columns
df_wide.columns = ['motif_name', 'cell_type', 'ACCESS', 'ATAC']
df_wide.to_csv(f"{out_dir}/fp_width.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sns.scatterplot(data=df_wide, x='ATAC', y='ACCESS', ax=ax, hue='cell_type')
ax.set_xlabel('Footprint Width (ATAC)')
ax.set_ylabel('Footprint Width (ACCESS)')
plt.plot([df_wide[['ACCESS', 'ATAC']].min().min(), df_wide[['ACCESS', 'ATAC']].max().max()], 
         [df_wide[['ACCESS', 'ATAC']].min().min(), df_wide[['ACCESS', 'ATAC']].max().max()], 'r--')